# Image clip embedding example


In [ ]:
import sys
import os
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook
from losses.image_image import ImageEmbeddingSimilarityLoss
from utils.train import train_with_criterion
from scenes import (
    SpringScene, SciFiRobotScene, BlenderManScene, CarScene, RedCarScene,
    CandleScene, HouseScene, DinoScene, FlowerPotScene, CarStudioScene,
    EinarScene, EinarSmallDomeScene, SpringPortraitScene, SpringPortraitSmallDomeScene,
)

In [ ]:
torch_precision = torch.float32
device = 'cuda' if torch.cuda.is_available() else 'cpu'

lr = 0.06
n_iter = 350
global_seed = 2  # Can be None
n_results = 4

# Scene registry (all common scenes)
scene_factories = {
    'SpringScene': lambda: SpringScene(device=device),
    'SciFiRobotScene': lambda: SciFiRobotScene(device=device),
    # 'BlenderManScene': lambda: BlenderManScene(device=device),
    'CarScene': lambda: CarScene(device=device),
    'RedCarScene': lambda: RedCarScene(device=device),
    'CandleScene': lambda: CandleScene(device=device),
    'HouseScene': lambda: HouseScene(device=device),
    'DinoScene': lambda: DinoScene(device=device),
    'FlowerPotScene': lambda: FlowerPotScene(device=device),
    'CarStudioScene': lambda: CarStudioScene(device=device),
    # 'EinarScene': lambda: EinarScene(device=device),
    'EinarSmallDomeScene': lambda: EinarSmallDomeScene(device=device),
    # 'SpringPortraitScene': lambda: SpringPortraitScene(device=device),
    'SpringPortraitSmallDomeScene': lambda: SpringPortraitSmallDomeScene(device=device),
}

# Select which scenes to run. By default this runs all scenes above.
selected_scene_names = list(scene_factories.keys())

scenes = [scene_factories[name]() for name in selected_scene_names]

color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())

In [ ]:
# Uses the checkpoint you requested by default
embedder_checkpoint = ""  # TODO: PATH_UPDATE image embedder checkpoint

model_name = 'vit_b_32'
similarity_mode = 'l2'  # 'cosine' or 'l2'

reference_image_paths = [
    "",  # TODO: PATH_UPDATE reference image path
    "",  # TODO: PATH_UPDATE reference image path
    "",  # TODO: PATH_UPDATE reference image path
]

output_subdirectory_name = 'image_clip_embedding_example'
run_all_reference_images = True

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

existing_reference_image_paths = [p for p in reference_image_paths if os.path.exists(p)]
missing_reference_image_paths = [p for p in reference_image_paths if not os.path.exists(p)]

if not os.path.exists(embedder_checkpoint):
    raise FileNotFoundError(f'Embedder checkpoint not found: {embedder_checkpoint}')

if len(existing_reference_image_paths) == 0:
    raise ValueError('No valid reference_image_paths found. Update the list before running training.')

if len(scenes) == 0:
    raise ValueError('No scenes selected. Update selected_scene_names before running training.')

if missing_reference_image_paths:
    print('Missing reference images (skipped):')
    for p in missing_reference_image_paths:
        print('  -', p)

print('Using embedder checkpoint:', embedder_checkpoint)
print('Found', len(existing_reference_image_paths), 'reference image(s).')
print('Selected scenes:', [scene.name for scene in scenes])

preview_count = min(3, len(existing_reference_image_paths))
fig, axes = plt.subplots(1, preview_count, figsize=(5 * preview_count, 4))
if preview_count == 1:
    axes = [axes]
for i in range(preview_count):
    img = Image.open(existing_reference_image_paths[i]).convert('RGB')
    axes[i].imshow(img)
    axes[i].set_title(os.path.basename(existing_reference_image_paths[i]))
    axes[i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
def run_image_embedding_similarity(scene, reference_image_path: str):
    width, height = scene.get_image_resolution()

    criterion = ImageEmbeddingSimilarityLoss(
        reference_image=reference_image_path,
        embedder_checkpoint=embedder_checkpoint,
        device=device,
        model_name=model_name,
        mode=similarity_mode,
    )

    image_name_no_ext = os.path.splitext(os.path.basename(reference_image_path))[0]
    title_prefix = f'ImageEmbeddingSimilarity ({similarity_mode}) - {scene.name} - {image_name_no_ext}'

    train_with_criterion(
        scene,
        lr, n_iter, criterion,
        starting_multiplier_std=(0.1, 0.1, 0.1),
        output_subdirectory_name=output_subdirectory_name,
        n_results=n_results,
        torch_precision=torch_precision,
        render_color_space_converter=color_space_converter,
        require_physically_plausible_multipliers=True,
        title_prefix=title_prefix,
        device=device,
        save_every=50,
        model_name=f'ImageEmbeddingSimilarity_{model_name}',
        pretrained_source=embedder_checkpoint,
        seed=global_seed,
        run_name_suffix=f'{scene.name}_{image_name_no_ext}',
    )

In [ ]:
if run_all_reference_images:
    for scene in scenes:
        for reference_image_path in existing_reference_image_paths:
            print('\n' + '=' * 70)
            print('Scene:', scene.name)
            print('Running reference image:', reference_image_path)
            print('=' * 70)
            run_image_embedding_similarity(scene, reference_image_path)
else:
    run_image_embedding_similarity(scenes[0], existing_reference_image_paths[0])